In [8]:
print("=== 1. DATASET SHAPE & INITIAL PROFILING ===")
import pandas as pd
import numpy as np

xl1 = pd.ExcelFile('Data/VELoop_Rewards_Data_01.xlsx')
xl2 = pd.ExcelFile('Data/VELOOP_Simulated_User_Behavior_1000.xlsx')

profile_rows = []
for wb_name, xl in [('VELoop_Rewards_Data_01.xlsx', xl1), ('VELOOP_Simulated_User_Behavior_1000.xlsx', xl2)]:
    for sheet in xl.sheet_names:
        df_temp = xl.parse(sheet)
        profile_rows.append({
            'Workbook': wb_name,
            'Sheet Name': sheet,
            'Rows': len(df_temp),
            'Columns': len(df_temp.columns),
            'Missing Values': int(df_temp.isnull().sum().sum()),
            'Duplicate Rows': int(df_temp.duplicated().sum())
        })

df_profile = pd.DataFrame(profile_rows)
print("=== STEP 1: WORKBOOK PROFILING SUMMARY ===")
display(df_profile)

# Load core sheets into memory for the cleaning steps
df_users_raw = xl1.parse('Users_Messy_Audit')
df_sessions = xl2.parse('02_Sessions')
df_events = xl2.parse('03_Feature_Events')
df_daily = xl2.parse('04_Daily_Activity')
df_tasks = xl2.parse('05_Task_Activity')
df_milestones = xl2.parse('06_Milestones')
df_features = xl2.parse('07_Feature_Usage')
df_audit_log = xl2.parse('10_Event_Data_Quality_Audit')

=== 1. DATASET SHAPE & INITIAL PROFILING ===


/tmp/ipykernel_3241/3164214132.py:11: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_temp = xl.parse(sheet)
/tmp/ipykernel_3241/3164214132.py:11: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_temp = xl.parse(sheet)
/tmp/ipykernel_3241/3164214132.py:11: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_temp = xl.parse(sheet)
/tmp/ipykernel_3241/3164214132.py:11: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_temp = xl.parse(sheet)
/tmp/ipykernel_3241/3164214132.py:11: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_temp = xl.parse(sheet)
/tmp/ipykernel_3241/3164214132.py:11: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_temp = xl.parse(sheet)
/tmp/ipykernel_3241/3164214132.py:11: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_temp = xl.parse(sheet)
/tmp/ipykernel_3241/

=== STEP 1: WORKBOOK PROFILING SUMMARY ===


,Workbook,Sheet Name,Rows,Columns,Missing Values,Duplicate Rows
0,VELoop_Rewards_Data_01.xlsx,Users_1000_Clean,1000,27,953,0
1,VELoop_Rewards_Data_01.xlsx,Users_Messy_Audit,1008,28,966,8
2,VELoop_Rewards_Data_01.xlsx,Data_Dictionary,25,5,0,0
3,VELoop_Rewards_Data_01.xlsx,Synthetic_Summary,8,2,0,0
4,VELoop_Rewards_Data_01.xlsx,README_Data_Status,7,2,0,0
5,VELOOP_Simulated_User_Behavior_1000.xlsx,01_Users_1000,1000,16,846,0
6,VELOOP_Simulated_User_Behavior_1000.xlsx,02_Sessions,7913,12,466,0
7,VELOOP_Simulated_User_Behavior_1000.xlsx,03_Feature_Events,35704,13,0,0
8,VELOOP_Simulated_User_Behavior_1000.xlsx,04_Daily_Activity,6667,17,0,0
9,VELOOP_Simulated_User_Behavior_1000.xlsx,05_Task_Activity,9846,12,10494,0


/tmp/ipykernel_3241/3164214132.py:26: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_users_raw = xl1.parse('Users_Messy_Audit')
/tmp/ipykernel_3241/3164214132.py:27: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_sessions = xl2.parse('02_Sessions')
/tmp/ipykernel_3241/3164214132.py:28: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_events = xl2.parse('03_Feature_Events')
/tmp/ipykernel_3241/3164214132.py:29: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_daily = xl2.parse('04_Daily_Activity')
/tmp/ipykernel_3241/3164214132.py:30: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_tasks = xl2.parse('05_Task_Activity')
/tmp/ipykernel_3241/3164214132.py:31: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df_milestones = xl2.parse('06_Milestones')
/tmp/ipykernel_3241/3164214132.py:32: Pandas4Warning: ExcelFil

In [2]:
print("=== STEP 2: REMOVING DUPLICATES ===")

df_users = df_users_raw.copy()

# Clean and deduplicate primary key (user_id) in Users table
raw_user_count = len(df_users)
df_users = df_users.drop_duplicates()
if 'user_id' in df_users.columns:
    df_users['user_id'] = df_users['user_id'].astype(str).str.strip().str.upper()
    df_users = df_users.drop_duplicates(subset=['user_id'], keep='first')

# Deduplicate activity and transaction logs
raw_task_count, raw_event_count = len(df_tasks), len(df_events)
df_tasks = df_tasks.drop_duplicates()
df_events = df_events.drop_duplicates()
df_daily = df_daily.drop_duplicates()
df_sessions = df_sessions.drop_duplicates()

print(f"Users: {raw_user_count} -> {len(df_users)} (Removed {raw_user_count - len(df_users)} duplicates)")
print(f"Tasks: {raw_task_count} -> {len(df_tasks)} (Removed {raw_task_count - len(df_tasks)} duplicates)")
print(f"Events: {raw_event_count} -> {len(df_events)} (Removed {raw_event_count - len(df_events)} duplicates)")

=== STEP 2: REMOVING DUPLICATES ===
Users: 1008 -> 1000 (Removed 8 duplicates)
Tasks: 9846 -> 9846 (Removed 0 duplicates)
Events: 35704 -> 35704 (Removed 0 duplicates)


In [3]:
print("=== STEP 3: STANDARDIZING TEXT & CATEGORIES ===")

# Strip whitespace and normalize fake null strings across all text columns
for col in df_users.select_dtypes(include='object').columns:
    df_users[col] = df_users[col].astype(str).str.strip()
    df_users[col] = df_users[col].replace({'nan': np.nan, 'None': np.nan, '': np.nan, 'null': np.nan, 'NULL': np.nan})

# Standardize Gender variations (M, male, MALE -> Male)
if 'gender' in df_users.columns:
    gender_map = {'M': 'Male', 'MALE': 'Male', 'F': 'Female', 'FEMALE': 'Female', 'O': 'Other', 'OTHER': 'Other'}
    df_users['gender'] = df_users['gender'].str.upper().map(lambda x: gender_map.get(x, x))

# Standardize Status, Country, Device, and Tier columns to clean Title Case
for col in df_users.select_dtypes(include='object').columns:
    if any(k in col.lower() for k in ['status', 'country', 'device', 'tier', 'source', 'platform']):
        df_users[col] = df_users[col].str.title()

# Standardize Task Status column in Tasks sheet
for col in df_tasks.select_dtypes(include='object').columns:
    if 'status' in col.lower() or 'state' in col.lower():
        df_tasks[col] = df_tasks[col].astype(str).str.strip().str.title()

print("Text standardization complete. Sample cleaned categories:")
if 'gender' in df_users.columns:
    print("Gender values:", df_users['gender'].dropna().unique())

=== STEP 3: STANDARDIZING TEXT & CATEGORIES ===
Text standardization complete. Sample cleaned categories:
Gender values: <StringArray>
['Male', 'Female', 'PREFER NOT TO SAY']
Length: 3, dtype: str


/tmp/ipykernel_3241/1023784319.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_users.select_dtypes(include='object').columns:
/tmp/ipykernel_3241/1023784319.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for detai

In [4]:
print("=== STEP 4: IMPUTING MISSING VALUES ===")
print("Missing values BEFORE imputation:\n", df_users.isnull().sum()[df_users.isnull().sum() > 0])

# 1. Fill specific categorical columns
if 'referral_source_detail' in df_users.columns:
    df_users['referral_source_detail'] = df_users['referral_source_detail'].fillna('Direct / Organic')

if 'gender' in df_users.columns:
    df_users['gender'] = df_users['gender'].fillna('Unspecified')

# 2. Fill any remaining text columns (excluding dates)
for col in df_users.select_dtypes(include='object').columns:
    if 'date' not in col.lower() and 'time' not in col.lower():
        df_users[col] = df_users[col].fillna('Unknown')

# 3. Fill numerical nulls (Age with median, financial/point columns with 0)
for col in df_users.select_dtypes(include=[np.number]).columns:
    if 'age' in col.lower():
        df_users[col] = df_users[col].fillna(df_users[col].median())
    else:
        df_users[col] = df_users[col].fillna(0)

print("\nMissing text/numeric values AFTER imputation:", df_users.select_dtypes(exclude=['datetime']).isnull().sum().sum())

=== STEP 4: IMPUTING MISSING VALUES ===
Missing values BEFORE imputation:
 age                            1
gender                         1
city                           1
registration_datetime          1
acquisition_source             1
referral_source_detail       881
primary_device                 1
email_verification_status      1
onboarding_status              1
last_login_datetime           73
dtype: int64

Missing text/numeric values AFTER imputation: 1


/tmp/ipykernel_3241/2765878141.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_users.select_dtypes(include='object').columns:


In [5]:
print("=== STEP 5: FIXING OUTLIERS & DATE LOGIC ===")

# 1. Fix Age outliers (valid range: 13 to 90)
if 'age' in df_users.columns:
    df_users['age'] = pd.to_numeric(df_users['age'], errors='coerce')
    valid_median_age = df_users.loc[(df_users['age'] >= 13) & (df_users['age'] <= 90), 'age'].median()
    outlier_ages = ((df_users['age'] < 13) | (df_users['age'] > 90)).sum()
    df_users.loc[(df_users['age'] < 13) | (df_users['age'] > 90), 'age'] = valid_median_age
    df_users['age'] = df_users['age'].fillna(valid_median_age).round(0).astype(int)
    print(f"Fixed {outlier_ages} age outliers (replaced with median age: {valid_median_age})")

# 2. Fix negative balances/points in all numeric columns
for col in df_users.select_dtypes(include=[np.number]).columns:
    if 'age' not in col.lower():
        neg_count = (df_users[col] < 0).sum()
        if neg_count > 0:
            print(f"Clipped {neg_count} negative values to 0 in column '{col}'")
            df_users[col] = df_users[col].clip(lower=0)

# 3. Parse Date columns and fix missing or chronological errors
date_cols = [c for c in df_users.columns if 'date' in c.lower() or 'time' in c.lower()]
for c in date_cols:
    df_users[c] = pd.to_datetime(df_users[c], errors='coerce')
    df_users[c] = df_users[c].fillna(df_users[c].median())

# Ensure last_login_datetime is never earlier than signup_date
signup_col = next((c for c in date_cols if 'signup' in c.lower() or 'created' in c.lower() or 'join' in c.lower()), None)
login_col = next((c for c in date_cols if 'login' in c.lower() or 'active' in c.lower()), None)
if signup_col and login_col:
    bad_dates = (df_users[login_col] < df_users[signup_col]).sum()
    df_users.loc[df_users[login_col] < df_users[signup_col], login_col] = df_users[signup_col]
    print(f"Corrected {bad_dates} records where {login_col} was earlier than {signup_col}")

print("Remaining Nulls Across Entire Users Table:", df_users.isnull().sum().sum())

=== STEP 5: FIXING OUTLIERS & DATE LOGIC ===
Fixed 0 age outliers (replaced with median age: 28.0)
Clipped 4 negative values to 0 in column 'days_since_registration'
Remaining Nulls Across Entire Users Table: 0


In [6]:
print("=== STEP 6: WALLET RECONCILIATION & USER-LEVEL KPI ROLLUPS ===")

# 1. Task Funnel Rollup per User
task_status_col = next((c for c in df_tasks.columns if 'status' in c.lower()), df_tasks.columns[-1])
task_rollup = df_tasks.groupby('user_id').agg(
    total_tasks_attempted=(task_status_col, 'count'),
    tasks_completed=(task_status_col, lambda x: (x == 'Completed').sum()),
    tasks_failed=(task_status_col, lambda x: (x != 'Completed').sum())
).reset_index()
task_rollup['task_completion_rate_pct'] = (
    (task_rollup['tasks_completed'] / task_rollup['total_tasks_attempted']) * 100
).round(2)

# 2. Retention Cohort Rollup per User (from 04_Daily_Activity)
if 'user_id' in df_daily.columns:
    retention_rollup = df_daily.groupby('user_id').size().reset_index(name='active_days')
    retention_rollup['retention_tier'] = pd.cut(
        retention_rollup['active_days'],
        bins=[-1, 1, 6, 13, 999],
        labels=['Churned (<=1d)', 'D1-D6 Active', 'D7-D13 Retained', 'D14+ Power User']
    )
    df_users = df_users.merge(retention_rollup, on='user_id', how='left')

# Merge Task Rollup into Users table
df_users = df_users.merge(task_rollup, on='user_id', how='left')
for c in ['total_tasks_attempted', 'tasks_completed', 'tasks_failed', 'task_completion_rate_pct', 'active_days']:
    if c in df_users.columns:
        df_users[c] = df_users[c].fillna(0)

# 3. Wallet Balance Reconciliation Check (Earned - Withdrawn vs Stored Balance)
earned_col = next((c for c in df_users.columns if 'earn' in c.lower() or 'credit' in c.lower()), None)
withdrawn_col = next((c for c in df_users.columns if 'withdraw' in c.lower() or 'payout' in c.lower() or 'redeem' in c.lower()), None)
balance_col = next((c for c in df_users.columns if 'balance' in c.lower() or 'wallet' in c.lower()), None)

if earned_col and withdrawn_col and balance_col:
    df_users['expected_wallet_balance'] = df_users[earned_col] - df_users[withdrawn_col]
    df_users['wallet_discrepancy'] = (df_users[balance_col] - df_users['expected_wallet_balance']).round(2)
    df_users['reconciliation_status'] = np.where(df_users['wallet_discrepancy'].abs() > 0.01, 'Mismatch Flagged', 'Reconciled')
    print("Wallet Reconciliation Breakdown:\n", df_users['reconciliation_status'].value_counts())
else:
    print("Numeric columns in df_users:", list(df_users.select_dtypes(include=[np.number]).columns))

display(df_users.head())

=== STEP 6: WALLET RECONCILIATION & USER-LEVEL KPI ROLLUPS ===
Numeric columns in df_users: ['age', 'active_days_x', 'xp_balance', 'days_since_registration', 'active_days_y', 'total_tasks_attempted', 'tasks_completed', 'tasks_failed', 'task_completion_rate_pct']


,user_id,full_name,email,age,gender,state,city,registration_datetime,acquisition_source,referral_source_detail,...,marketing_opt_in,account_created_via,country,data_environment,active_days_y,retention_tier,total_tasks_attempted,tasks_completed,tasks_failed,task_completion_rate_pct
0,VLUSR000001,Sanjay Singh,sanjay.singh1@gmail.com,35,Male,Odisha,Unknown,2026-06-10 07:14:00,Organic Search,Direct / Organic,...,No,Web Registration,India,SIMULATED — NOT ACTUAL VELOOP DATA,1.0,Churned (<=1d),6.0,1.0,5.0,16.67
1,VLUSR000002,Diya Chatterjee,diya.chatterjee2@gmail.com,20,Female,Odisha,Bhubaneswar,2026-05-27 14:37:00,Unknown,Direct / Organic,...,Yes,Web Registration,India,SIMULATED — NOT ACTUAL VELOOP DATA,4.0,D1-D6 Active,9.0,5.0,4.0,55.56
2,VLUSR000003,Pooja Singh,pooja.singh3@outlook.com,20,Male,Bihar,Gaya,2026-09-02 08:51:00,Organic Search,Direct / Organic,...,No,Web Registration,India,SIMULATED — NOT ACTUAL VELOOP DATA,9.0,D7-D13 Retained,11.0,7.0,4.0,63.64
3,VLUSR000004,Nikhil Das,nikhil.das4@outlook.com,35,PREFER NOT TO SAY,Uttar Pradesh,Noida,2026-04-21 07:55:00,Organic Search,Direct / Organic,...,No,Google Sign-In,India,SIMULATED — NOT ACTUAL VELOOP DATA,3.0,D1-D6 Active,13.0,7.0,6.0,53.85
4,VLUSR000005,Aditi Patel,aditi.patel5@yahoo.com,24,PREFER NOT TO SAY,West Bengal,Durgapur,2026-06-09 20:44:00,Social Media,Direct / Organic,...,Yes,Google Sign-In,India,SIMULATED — NOT ACTUAL VELOOP DATA,1.0,Churned (<=1d),8.0,6.0,2.0,75.00


In [7]:
print("=== STEP 7: EXPORTING FINAL MASTER WORKBOOK FOR POWER BI ===")

output_path = 'VELoop_PowerBI_Master_Dataset.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_users.to_excel(writer, sheet_name='01_Users_Cleaned_Master', index=False)
    df_sessions.to_excel(writer, sheet_name='02_Sessions_Cleaned', index=False)
    df_events.to_excel(writer, sheet_name='03_Events_Cleaned', index=False)
    df_daily.to_excel(writer, sheet_name='04_Daily_Activity_Cleaned', index=False)
    df_tasks.to_excel(writer, sheet_name='05_Tasks_Cleaned', index=False)
    df_milestones.to_excel(writer, sheet_name='06_Milestones_Cleaned', index=False)
    df_features.to_excel(writer, sheet_name='07_Feature_Usage_Cleaned', index=False)
    df_audit_log.to_excel(writer, sheet_name='08_Data_Quality_Audit', index=False)

# Also update the single CSV file in your repo root
df_users.to_csv('cleaned_users_audit_output.csv', index=False)

print(f"✅ All 8 sheets cleaned and exported to '{output_path}'!")

=== STEP 7: EXPORTING FINAL MASTER WORKBOOK FOR POWER BI ===
✅ All 8 sheets cleaned and exported to 'VELoop_PowerBI_Master_Dataset.xlsx'!
